# 🤖 VYNERA AI — Training Notebook (Fixed)
**Kaggle · GPU T4 x2 · DialoGPT-medium**

### Fixes in this version:
- `loss=nan` fixed — model loaded in `float32`, AMP handles fp16 internally via `torch.cuda.amp`
- Gradient scaler added to prevent fp16 underflow
- Model saved immediately after first epoch so inference always has a checkpoint
- DataParallel loss averaging fixed
- Inference test uses the correct saved path

### Before running:
1. Upload `train.jsonl` and `val.jsonl` via **Add Data → Upload**
2. Accelerator → **GPU T4 x2** is enabled
3. Internet → **On**
4. Run all cells top to bottom

## Step 0 — Find uploaded JSONL files

In [ ]:
import os, glob

def find_file(name):
    for root in ['/kaggle/input', '.', '/kaggle/working']:
        for match in glob.glob(f'{root}/**/{name}', recursive=True):
            return match
    return None

TRAIN_FILE = find_file('train.jsonl')
VAL_FILE   = find_file('val.jsonl')

print(f'train.jsonl → {TRAIN_FILE}')
print(f'val.jsonl   → {VAL_FILE}')

assert TRAIN_FILE, '❌ train.jsonl not found! Upload via Add Data → Upload Dataset'
assert VAL_FILE,   '❌ val.jsonl not found!   Upload via Add Data → Upload Dataset'

train_count = sum(1 for _ in open(TRAIN_FILE))
val_count   = sum(1 for _ in open(VAL_FILE))
print(f'\n✅ Train pairs : {train_count:,}')
print(f'✅ Val pairs   : {val_count:,}')

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install transformers==4.41.1 accelerate==0.30.0 sentencepiece==0.2.0 -q

## Step 2 — Verify GPU

In [ ]:
import torch

print(f'PyTorch        : {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU count      : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB')

assert torch.cuda.is_available(), '❌ No GPU! Enable T4 x2 in Settings → Accelerator'

## Step 3 — Config

In [ ]:
import json, os

BASE_MODEL  = 'microsoft/DialoGPT-medium'
OUTPUT_DIR  = '/kaggle/working/models'
MAX_LEN     = 256
BATCH_SIZE  = 4        # per GPU — lowered to prevent OOM with float32 AMP
GRAD_ACCUM  = 8        # effective batch = 4 x 2 GPUs x 8 = 64
EPOCHS      = 3
LR          = 3e-5     # slightly lower — safer with AMP
WARMUP_RATIO= 0.1

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir : {OUTPUT_DIR}')
print(f'Batch/GPU  : {BATCH_SIZE}  |  Grad accum: {GRAD_ACCUM}  |  Effective batch: {BATCH_SIZE * 2 * GRAD_ACCUM}')

## Step 4 — Load DialoGPT-medium (float32 — AMP handles fp16 internally)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading: {BASE_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ⚠️  Load in float32 — GradScaler + AMP autocast handles fp16 during forward/backward
# Loading directly in float16 with DataParallel causes NaN gradients on T4
device = 'cuda'
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,   # float32 weights — AMP casts internally
).to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\n✅ Model on {device}  |  Parameters: {n_params:.1f}M  |  dtype: {next(model.parameters()).dtype}')

## Step 5 — Dataset and DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader

class VyneraDataset(Dataset):
    def __init__(self, path):
        self.samples = []
        EOS = tokenizer.eos_token
        skipped = 0
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                    inp = str(row.get('input',  '')).strip()
                    out = str(row.get('output', '')).strip()
                    if inp and out:
                        self.samples.append(f'{inp}{EOS}{out}{EOS}')
                    else:
                        skipped += 1
                except Exception:
                    skipped += 1
        print(f'  Loaded {len(self.samples):,} samples  (skipped {skipped})')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.samples[idx],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt',
        )
        ids = enc['input_ids'].squeeze()
        return {'input_ids': ids, 'labels': ids.clone()}


def collate_fn(batch):
    ml  = max(b['input_ids'].size(0) for b in batch)
    inp = torch.zeros(len(batch), ml, dtype=torch.long)
    lbl = torch.full((len(batch), ml), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = b['input_ids'].size(0)
        inp[i, :L] = b['input_ids']
        lbl[i, :L] = b['labels']
    return {
        'input_ids':      inp,
        'attention_mask': (inp != 0).long(),
        'labels':         lbl,
    }


print('Building datasets...')
train_ds = VyneraDataset(TRAIN_FILE)
val_ds   = VyneraDataset(VAL_FILE)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'\n✅ Train batches: {len(train_dl):,}  |  Val batches: {len(val_dl):,}')

## Step 6 — Multi-GPU + GradScaler (fixes NaN loss)

In [ ]:
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f'✅ DataParallel across {n_gpus} GPUs')
else:
    print('Single GPU mode')

model = model.to(device)

# GradScaler prevents fp16 gradient underflow during AMP autocast
scaler = GradScaler()
print('✅ GradScaler initialised (prevents NaN gradients in fp16 autocast)')

## Step 7 — Training loop (AMP + GradScaler)

In [ ]:
import gc, math
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = (len(train_dl) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_val_loss  = float('inf')
train_losses   = []
val_losses     = []
model_saved    = False

print(f'Steps/epoch: {len(train_dl):,}  |  Total optim steps: {total_steps:,}  |  Warmup: {warmup_steps}')
print('=' * 68)

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    running_loss  = 0.0
    valid_steps   = 0
    optimizer.zero_grad()

    pbar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS} [train]')
    for step, batch in enumerate(pbar, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        # AMP autocast: forward pass runs in fp16, weights stay float32
        with autocast():
            outputs = model(**batch)
            loss    = outputs.loss
            # DataParallel returns per-GPU losses — average them
            if loss.dim() > 0:
                loss = loss.mean()

        # Skip NaN/Inf — shouldn't happen with scaler but just in case
        if not math.isfinite(loss.item()):
            print(f'  ⚠ Non-finite loss at step {step}: {loss.item():.4f} — skipping')
            optimizer.zero_grad()
            scaler.update()
            continue

        scaler.scale(loss / GRAD_ACCUM).backward()
        running_loss += loss.item()
        valid_steps  += 1

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 500 == 0 and valid_steps > 0:
            avg = running_loss / valid_steps
            pbar.set_postfix({'loss': f'{avg:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}',
                              'scale': f'{scaler.get_scale():.0f}'})

    avg_train = running_loss / max(valid_steps, 1)
    train_losses.append(avg_train)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss    = 0.0
    val_steps   = 0
    with torch.no_grad():
        for batch in tqdm(val_dl, desc=f'Epoch {epoch}/{EPOCHS} [val]  ', leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast():
                out = model(**batch)
                l   = out.loss
                if l.dim() > 0:
                    l = l.mean()
            if math.isfinite(l.item()):
                val_loss  += l.item()
                val_steps += 1

    avg_val = val_loss / max(val_steps, 1)
    val_losses.append(avg_val)

    print(f'\nEpoch {epoch}  |  Train loss: {avg_train:.4f}  |  Val loss: {avg_val:.4f}  |  GradScale: {scaler.get_scale():.0f}')

    # Always save the best checkpoint (and at least one after epoch 1)
    if avg_val < best_val_loss or not model_saved:
        best_val_loss = avg_val
        core = model.module if hasattr(model, 'module') else model
        core.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        model_saved = True
        print(f'  ✅ Saved checkpoint  (val loss: {avg_val:.4f})')

    gc.collect()
    torch.cuda.empty_cache()

print(f'\n🎉 Training complete!  Best val loss: {best_val_loss:.4f}')
print(f'Model saved to: {OUTPUT_DIR}')

# Verify save
saved_files = os.listdir(OUTPUT_DIR)
print(f'Saved files: {saved_files}')
assert 'config.json' in saved_files, '❌ config.json missing — model was not saved!'
print('✅ Model save verified.')

## Step 8 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = list(range(1, len(train_losses) + 1))

# Check for NaN before plotting
has_valid = any(math.isfinite(v) for v in train_losses + val_losses)
if not has_valid:
    print('⚠ All losses are NaN/Inf — training did not converge. Check data and rerun.')
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs_range, train_losses, 'o-', label='Train loss', color='#C9FF00', linewidth=2, markersize=8)
    ax.plot(epochs_range, val_losses,   's-', label='Val loss',   color='#00BFFF', linewidth=2, markersize=8)
    for i, (t, v) in enumerate(zip(train_losses, val_losses)):
        ax.annotate(f'{t:.3f}', (epochs_range[i], t), textcoords='offset points', xytext=(0, 10),
                    ha='center', color='#C9FF00', fontsize=9)
        ax.annotate(f'{v:.3f}', (epochs_range[i], v), textcoords='offset points', xytext=(0, -16),
                    ha='center', color='#00BFFF', fontsize=9)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.set_title('VYNERA AI — Training Curves', fontsize=13)
    ax.set_xticks(epochs_range)
    ax.legend()
    ax.set_facecolor('#111')
    fig.patch.set_facecolor('#000')
    for spine in ax.spines.values(): spine.set_edgecolor('#444')
    ax.tick_params(colors='#aaa')
    ax.yaxis.label.set_color('#aaa'); ax.xaxis.label.set_color('#aaa'); ax.title.set_color('#eee')
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved training_curves.png')
    print(f'Final train loss: {train_losses[-1]:.4f}  |  Final val loss: {val_losses[-1]:.4f}')

## Step 9 — Inference test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Verify the directory has required files before loading
required = ['config.json', 'tokenizer.json']
missing  = [f for f in required if not os.path.exists(os.path.join(OUTPUT_DIR, f))]
if missing:
    print(f'❌ Missing files in {OUTPUT_DIR}: {missing}')
    print('Training may have failed. Check Step 7 output.')
else:
    print(f'Loading saved model from {OUTPUT_DIR} ...')
    test_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
    test_mdl = AutoModelForCausalLM.from_pretrained(
        OUTPUT_DIR, torch_dtype=torch.float16
    ).to('cuda').eval()

    EOS = test_tok.eos_token

    def chat(msg, max_new=150):
        ids = test_tok.encode(f'{msg}{EOS}', return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = test_mdl.generate(
                ids,
                max_new_tokens=max_new,
                do_sample=True,
                temperature=0.85,
                top_p=0.92,
                repetition_penalty=1.3,
                pad_token_id=test_tok.eos_token_id,
                eos_token_id=test_tok.eos_token_id,
            )
        return test_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

    tests = [
        'I feel really overwhelmed today.',
        'I just got a new job and I am so excited!',
        'I have been feeling anxious about my future.',
        'I feel like nobody understands me.',
        'I feel empty inside and I do not know why.',
    ]

    print('\n' + '=' * 64)
    print('INFERENCE TEST')
    print('=' * 64)
    for prompt in tests:
        resp = chat(prompt)
        print(f'User   : {prompt}')
        print(f'VYNERA : {resp}')
        print('-' * 64)

## Step 10 — List files and zip for download

In [ ]:
import os, shutil

print(f'Files in {OUTPUT_DIR}:')
total_mb = 0
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    total_mb += size_mb
    print(f'  {fname:<45}  {size_mb:>7.1f} MB')
print(f'  Total: {total_mb:.1f} MB')

zip_path = '/kaggle/working/vynera_model'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(zip_path + '.zip') / 1e6
print(f'\n✅ vynera_model.zip  ({zip_size:.0f} MB)')
print('   → Output panel (right side) → vynera_model.zip → ⋮ → Download')